# Error analysis: what does DistilBERT still get wrong?

Run `python -m src.evaluate_test` first; it creates `results/distilbert_errors.csv`.

**Categories to tag** (add your own if you find new ones):

| Tag | Meaning |
|---|---|
| `mixed` | Positive and negative in the same review ("worked, but side effects...") |
| `label_noise` | The text clearly disagrees with the star rating; the model is arguably right |
| `truncated` | The review is cut off mid-sentence in the dataset |
| `too_short` | Too little text to judge |
| `context` | Needs outside knowledge ("wish they'd bring it back") |
| `sarcasm` | Says the opposite of what it means |
| `model_wrong` | Clear review, the model just got it wrong |

In [2]:
import pandas as pd

pd.set_option("display.max_colwidth", None)   # show the full review text

errors = pd.read_csv("../results/distilbert_errors.csv")
test = pd.read_csv("../data/processed/test.csv")
errors = errors.merge(test[["text", "rating"]].drop_duplicates("text"), on="text", how="left")
print(f"{len(errors):,} mistakes")
errors["label"].value_counts().rename({0: "true negative, predicted positive", 1: "true positive, predicted negative"})

2,635 mistakes


label
true negative, predicted positive    1494
true positive, predicted negative    1141
Name: count, dtype: int64

## 1. A random sample of 50 mistakes to read

In [5]:
sample = errors.sample(50, random_state=42).reset_index(drop=True)
sample[["rating", "label", "prediction", "text"]]

,rating,label,prediction,text
0,4,0,1,When I first got nexplanon in my arm hurt for about two years. Also it makes my period about 32 days with having about 60 day break. I don't care much for it in that regard. It does work good as far as not having any babies.
1,10,1,0,Took it for a year or so but now I need it again and my insurance wont pay for it and my other medicines are over $400.00 a month and they want $160.00 more for Crestor. We just can't do it.
2,1,0,1,I have been on this pill for about 6 months face is nice and clear but that's it.
3,2,0,1,Knocked out the (dental) infection FAST.
4,10,1,0,For sure this pill does what it's supposed to do. But after taking this I had breakouts on my chin area. Also I was late for my period.
5,1,0,1,I have bipolar 2 but was misdiagnosed with major depression for over 20 years.
6,1,0,1,After being diagnosed with bipolar disorder my doctor put me on this medication.
7,2,0,1,I was on Yasmin for almost 6 years and it worked wonders for me I never suffered with bad skin. I was prescribed Yasmin for heavy bleeding on my periods and it helped a lot with my flow of bleeding. I never noticed any mood changes or negative things to say about Yasmin pill. A few days ago I was diagnosed with low serotonin levels in my brain ( Neurotransmitter responsible for your moods ) doctors and pharmacists say it
8,10,1,0,I do not know about other medication to alleviate BPD symptoms. So I have nothing to compare Abilify with. I've been taking Abilify 10mg for over 2 years. Doing great I decided to stop it. Within a few days BPD symptoms came back.... mood swings rage. I'm back on Abilify and I feel much better again. It was reckless of me to think I could do without.
9,1,0,1,I've been taking it regularly for about a week it was very helpful at first and sleep well.


## 2. Tag them

Read each review above, then write its tag below, in the same order (row 0 first).

In [6]:
tags = [
    "mixed", "context", "mixed", "label_noise", "mixed", "context", "context", "truncated", "context", "label_noise",
      "too_short", "mixed", "too_short", "too_short", "truncated", "too_short", "truncated", "truncated", "too_short", "context",
      "truncated", "mixed", "mixed", "model_wrong", "label_noise", "context", "truncated", "label_noise", "mixed", "context",
      "too_short", "mixed", "model_wrong", "model_wrong", "mixed", "label_noise", "scrape_artifact", "context", "context", "model_wrong",
      "mixed", "model_wrong", "negation", "mixed", "mixed", "context", "context", "mixed", "mixed", "too_short"
]

sample["tag"] = pd.Series(tags, dtype="object")
sample["tag"].value_counts()

tag
mixed              14
context            11
too_short           7
truncated           6
label_noise         5
model_wrong         5
scrape_artifact     1
negation            1
Name: count, dtype: int64

## 3. Save your tagged sample (for the README)

In [7]:
sample[["rating", "label", "prediction", "tag", "text"]].to_csv("../results/error_analysis_sample.csv", index=False)

## 4. Your findings (write 3-5 bullet points here)

-  Mixed sentiment was a major source of errors. Many reviews mentioned both positive effects and negative side effects, making it difficult for the model to determine the overall sentiment.

- The model struggled with context. Some reviews described medical conditions, past experiences, or reasons for taking medication without clearly expressing an opinion about the drug.

- Short and incomplete reviews caused misclassifications. Reviews with very little information or truncated sentences lacked enough context for the model to make accurate predictions.

- Some errors were caused by label noise. Certain reviews appeared to have incorrect or ambiguous ground-truth labels, meaning the model's prediction was sometimes reasonable despite being marked incorrect.

- The model had difficulty understanding nuanced language. Negation, comparisons between medications, and statements about treatment effectiveness sometimes led to incorrect predictions, suggesting that the model relies too heavily on individual sentiment-related words rather than the overall meaning.